<a href="https://colab.research.google.com/github/sponfork/HnC-Classifier/blob/main/Heart_Lung_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#main
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa

!cp -r /content/drive/MyDrive/HLC_dataset/HLS-CMDS /content/HLS-CMDS

class HLSCMDSDataset(Dataset):
    def __init__(self, csv_file, audio_dir, id_column="Heart Sound ID", label_column="Heart Sound Type", target_sr=16000, segment_duration=3.0):
        self.metadata = pd.read_csv(csv_file)
        self.audio_dir = audio_dir
        self.id_column = id_column
        self.label_column = label_column
        self.target_sr = target_sr
        self.samples_per_segment = int(target_sr * segment_duration)

        unique_labels = self.metadata[self.label_column].unique()
        self.label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
        print(f"Discovered labels for {label_column}: {self.label_mapping}")

        self.samples_index = []
        self._prepare_dataset()

    def _prepare_dataset(self):
        for _, row in self.metadata.iterrows():
            file_id = row[self.id_column]
            filename = f"{file_id}.wav"

            label_str = row[self.label_column]
            label = self.label_mapping.get(label_str, 0)

            file_path = os.path.join(self.audio_dir, filename)
            if not os.path.exists(file_path):
                continue

            y, sr = librosa.load(file_path, sr=self.target_sr)
            total_samples = len(y)
            num_segments = total_samples // self.samples_per_segment

            for seg_idx in range(num_segments):
                self.samples_index.append({
                    'file_path': file_path,
                    'start_sample': seg_idx * self.samples_per_segment,
                    'label': label
                })

    def __len__(self):
        return len(self.samples_index)

    def __getitem__(self, idx):
        item = self.samples_index[idx]

        y, _ = librosa.load(
            item['file_path'],
            sr=self.target_sr,
            offset=item['start_sample'] / self.target_sr,
            duration=3.0
        )

        mel_spectrogram = librosa.feature.melspectrogram(
            y=y,
            sr=self.target_sr,
            n_mels=128,
            n_fft=2048,
            hop_length=512
        )

        log_mel_spec = librosa.power_to_db(mel_spectrogram, ref=np.max)

        audio_tensor = torch.tensor(log_mel_spec, dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(item['label'], dtype=torch.long)

        return audio_tensor, label_tensor

base_dir = "/content/HLS-CMDS"

heart_dataset = HLSCMDSDataset(
    csv_file=f"{base_dir}/HS.csv",
    audio_dir=f"{base_dir}/HS/HS",
    id_column="Heart Sound ID",
    label_column="Heart Sound Type"
)

heart_loader = DataLoader(heart_dataset, batch_size=32, shuffle=True, num_workers=0)

def train_audio_classifier(model, dataloader, num_epochs=5, learning_rate=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"\nTraining started on device: {device}\n", flush=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for batch_idx, (audio_tensor, label_tensor) in enumerate(dataloader):

            audio_tensor = audio_tensor.to(device)
            label_tensor = label_tensor.to(device)

            optimizer.zero_grad()

            outputs = model(audio_tensor)

            loss = criterion(outputs, label_tensor)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * audio_tensor.size(0)
            _, predicted = torch.max(outputs, dim=1)
            correct_predictions += (predicted == label_tensor).sum().item()
            total_samples += label_tensor.size(0)

            if (batch_idx + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] | Batch [{batch_idx+1}/{len(dataloader)}] | Batch Loss: {loss.item():.4f}", flush=True)

        epoch_loss = running_loss / total_samples
        epoch_acc = correct_predictions / total_samples

        print(f"=== EPOCH {epoch+1} FINISHED ===")
        print(f"Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc * 100:.2f}%\n", flush=True)

    print("Training loop completed successfully!")
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
heart_model = LightweightAudioCNN(num_classes=10).to(device)
trained_model = train_audio_classifier(heart_model, heart_loader, num_epochs=20)
torch.optim.lr_scheduler.ReduceLROnPlateau

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LightweightAudioCNN(nn.Module):
    def __init__(self, num_classes):
        super(LightweightAudioCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.global_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
#main
import os
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset, DataLoader

base_dir = "/content/drive/MyDrive/HLC_dataset/HLS-CMDS"

heart_dataset = HLSCMDSDataset(
    csv_file=f"{base_dir}/HS.csv",
    audio_dir=f"{base_dir}/HS/HS",
    id_column="Heart Sound ID",
    label_column="Heart Sound Type"
)

heart_loader = DataLoader(
    heart_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

print(f"Success! 'heart_dataset' is now loaded with {len(heart_dataset)} segments.")

In [ ]:
import matplotlib.pyplot as plt
import librosa.display

audio_tensor, label_tensor = heart_dataset[0]

spec_to_plot = audio_tensor.squeeze(0).numpy()

plt.figure(figsize=(10, 4))
librosa.display.specshow(
    spec_to_plot,
    sr=16000,
    hop_length=512,
    x_axis='time',
    y_axis='mel',
    cmap='viridis'
)
plt.colorbar(format='%+2.0f dB')
plt.title(f"Log-Mel Spectrogram | Label Index: {label_tensor.item()}")
plt.tight_layout()
plt.show()